In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('songdata.csv')
df.head(14)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...
5,ABBA,Burning My Bridges,/a/abba/burning+my+bridges_20003011.html,"Well, you hoot and you holler and you make me ..."
6,ABBA,Cassandra,/a/abba/cassandra_20002811.html,Down in the street they're all singing and sho...
7,ABBA,Chiquitita,/a/abba/chiquitita_20002978.html,"Chiquitita, tell me what's wrong \r\nYou're e..."
8,ABBA,Crazy World,/a/abba/crazy+world_20003013.html,I was out with the morning sun \r\nCouldn't s...
9,ABBA,Crying Over You,/a/abba/crying+over+you_20177611.html,I'm waitin' for you baby \r\nI'm sitting all ...


In [3]:
df.shape

(57650, 4)

In [4]:
df = df.sample(n=5000).drop('link', axis=1).reset_index(drop=True)

In [5]:
df.shape

(5000, 3)

In [6]:
df['text'] = df['text'].str.lower().replace(r'[^\w\s]','').replace(r'\n',' ', regex=True)

In [7]:
df['text'][0]

"they call her the lady of the night  \r she's a woman of the world  \r and easy-living girl with love for sale  \r   \r that's what they call her the lady of the night  \r no one seems to know her name  \r and even less about the place  \r from where she came  \r   \r every evening when the night is close at hand  \r you'll find the lady on the rue d'avignon  \r in a half lit hotel doorway the lady advertises warmly  \r it's just a job but she'll do the best she can  \r   \r don't try to change this lady of the night  \r she's a lot like you and me  \r and less than what she seems to be  \r   \r (she is the lady of the night)  \r and easy-living girl  \r (she is the lady of the night)  \r she's a woman of the world  \r (she is the lady of the night)  \r with lots of loving for sale  \r (she is the lady of the night)  \r lady lady of the night  \r   \r round here they call her the lady of the night  \r in a perfume hotel room  \r shadows dance upon the wall and fate at dawn  \r she's n

In [9]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenization(txt):
    tokens = nltk.word_tokenize(txt)
    stemming = [stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

In [10]:
df['text'] = df['text'].apply(lambda x: tokenization(x))

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
tfidvector = TfidfVectorizer(analyzer='word',stop_words='english')
matrix = tfidvector.fit_transform(df['text'])
similarity = cosine_similarity(matrix)

In [14]:
similarity[0]

array([1.        , 0.00754902, 0.03085276, ..., 0.01581547, 0.02129799,
       0.05919299])

In [15]:
df[df['song']=='Bang'].index[0]

4547

# recommedation function

In [16]:
def recommendation(song_df):
    idx = df[df['song'] == song_df].index[0]
    distances = sorted(list(enumerate(similarity[idx])),reverse=True,key=lambda x:x[1])
    
    songs = []
    for m_id in distances[1:21]:
        songs.append(df.iloc[m_id[0]].song)
        
    return songs

In [17]:
recommendation('Bang')

['The Prime Of Your Love',
 'Give Me A Bit',
 'Solsbury Hill',
 'Bang Bang',
 'Bang Bang Bang',
 'Shoot From The Hip',
 'Pop That Thang',
 'Just Like Forever',
 "Nothin' Else",
 'Show Me',
 'Funky Music Sho Nuff Turns Me On',
 'I Saw Her Standing There',
 'Loved',
 'Love Me Tender',
 'The Way You Move',
 'Another Day',
 'Who Will Love Me Now',
 'Learn To Love',
 "I Think I'm In Love",
 'Birthday Song']

In [18]:
import pickle
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('df.pkl','wb'))